In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
file_path = "dataset/Wlift.csv"
data = pd.read_csv(file_path)

# Print available columns for debugging
print("\n📌 Available Columns in Dataset:", data.columns.tolist())

# Define Features (Ensure only available columns are included)
selected_features = ["age", "age_start", "yrs_experience", "sex_encoded", 
                     "body_weight", "lifted_weight", "shoulder_angle", 
                     "knees_angle", "back_angle", "wrist_angle", "hips_angle"]

# Check if "performance" column exists
if "performance" not in data.columns:
    raise KeyError("❌ 'performance' column is missing from dataset!")

# Define Fatigue Labels (High = 0, Moderate = 1, Low = 2)
data["fatigue_label"] = np.where(data["performance"] < 50, 0,  # High Fatigue
                        np.where(data["performance"] < 80, 1,  # Moderate Fatigue
                        2))  # Low Fatigue

# Prepare Data
X = data[selected_features]
y = data["fatigue_label"]

# Train-test split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save the scaler
joblib.dump(scaler, "fatigue_scaler.pkl")

# Define Models for Evaluation
models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(kernel="rbf", probability=True)
}

# Train and Evaluate Models
best_model = None
best_accuracy = 0

for model_name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    
    print(f"\n📌 Model: {model_name}")
    print(f"✅ Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, y_pred))
    
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_model = model

# Save Best Model
joblib.dump(best_model, "fatigue_model.pkl")
print(f"\n🎯 Best Model Saved: {best_model.__class__.__name__} with Accuracy: {best_accuracy:.4f}")



📌 Available Columns in Dataset: ['id', 'sex', 'age', 'agegrp3', 'age_start', 'yrs_experience', 'shoulder', 'knees', 'back', 'wrist', 'hips', 'OA', 'train_days', 'train_session', 'train_warm', 'train_lift', 'train_strength', 'train_supp', 'train_cool', 'pcoach', 'premote', 'pown', 'nutrition', 'pa_power', 'pa_body', 'pa_cf', 'pa_ball', 'pa_fit', 'pa_endure', 'pa_track', 'pa_ma', 'pa_yoga', 'sport0_power', 'sport0_body', 'sport0_cf', 'sport0_ball', 'sport0_fit', 'sport0_endure', 'sport0_track', 'sport0_ma', 'sport0_yoga', 'sport0_gym', 'sport0_strength', 'sport0_impact', 'body_weight', 'weight_class', 'performance', 'shoulder_angle', 'knees_angle', 'back_angle', 'wrist_angle', 'hips_angle', 'lifted_weight', 'sex_encoded', 'max_performance']

📌 Model: RandomForest
✅ Accuracy: 0.8571
              precision    recall  f1-score   support

           0       0.84      0.69      0.76        59
           1       0.83      0.92      0.88       103
           2       1.00      0.96      0.98  